# Build the CertVIC CPython 3.12 offline wheelhouse

Settings: **Accelerator OFF**, **Internet ON**. Attach the authenticated CertVIC CODE, CONFIGS, and EXECUTION_TOOLS inputs. Run All without editing. This notebook produces `certvic_offline_wheelhouse_cp312.zip`; it does not produce scientific evidence.


In [ ]:
import json, os, pathlib, platform, shutil, stat, sys, zipfile
from packaging.tags import sys_tags
probe = {"executable": sys.executable, "implementation": platform.python_implementation(), "python": platform.python_version(), "architecture": platform.machine(), "system": platform.system(), "libc": platform.libc_ver(), "supported_tags": [str(tag) for tag in sys_tags()]}
print(json.dumps({"status": "IMMEDIATE_CP312_PROVISIONING_PROBE", **probe}, indent=2))
if probe["implementation"] != "CPython" or not probe["python"].startswith("3.12.") or probe["architecture"].lower() != "x86_64" or probe["system"] != "Linux":
    raise RuntimeError("CERTVIC_RUNTIME_01_PYTHON_PROFILE_NOT_SUPPORTED: builder requires Kaggle CPython 3.12 Linux x86_64")
if not probe["libc"][0].lower().startswith("glibc") or tuple(map(int, probe["libc"][1].split("."))) < (2, 17):
    raise RuntimeError("CERTVIC_RUNTIME_01_PYTHON_PROFILE_NOT_SUPPORTED: glibc >= 2.17 required")


In [ ]:
roots = [pathlib.Path(value) for value in os.environ.get("CERTVIC_INPUT_ROOTS", "/kaggle/input").split(os.pathsep) if pathlib.Path(value).is_dir()]
project_candidates = sorted({path.parent.resolve() for root in roots for path in root.rglob("pyproject.toml") if (path.parent / "certvic/__init__.py").is_file()})
if not project_candidates:
    extraction = pathlib.Path("/kaggle/working/certvic_provisioning_code")
    extraction.mkdir(parents=True, exist_ok=True)
    for root in roots:
        for archive_path in root.rglob("*"):
            if not archive_path.is_file() or archive_path.is_symlink(): continue
            try:
                with zipfile.ZipFile(archive_path) as archive:
                    names = [info.filename for info in archive.infolist()]
                    if not any(name.endswith("pyproject.toml") for name in names) or not any(name.endswith("certvic/__init__.py") for name in names): continue
                    if len(names) != len(set(names)) or archive.testzip() is not None: raise RuntimeError("invalid CODE archive")
                    for info in archive.infolist():
                        value = pathlib.PurePosixPath(info.filename)
                        mode = (info.external_attr >> 16) & 0xFFFF
                        if value.is_absolute() or ".." in value.parts or stat.S_ISLNK(mode): raise RuntimeError("unsafe CODE archive")
                    archive.extractall(extraction)
            except zipfile.BadZipFile:
                continue
    project_candidates = sorted({path.parent.resolve() for path in extraction.rglob("pyproject.toml") if (path.parent / "certvic/__init__.py").is_file()})
if len(project_candidates) != 1: raise RuntimeError(f"authenticated CODE project discovery ambiguous: {project_candidates}")
PROJECT_ROOT = project_candidates[0]
sys.path.insert(0, str(PROJECT_ROOT))
from certvic.cvpr.content_discovery import discover_authenticated_input
from certvic.cvpr.notebook_bootstrap import discover_unique_file
from certvic.cvpr.wheelhouse_builder import deterministic_provision
materialized = pathlib.Path("/kaggle/working/certvic_provisioning_inputs")
code = discover_authenticated_input("CODE", roots=roots, materialization_root=materialized)
configs = discover_authenticated_input("CONFIGS", roots=roots, materialization_root=materialized)
tools = discover_authenticated_input("EXECUTION_TOOLS", roots=roots, materialization_root=materialized)
print({"authenticated_content_identities": {row["role"]: row["content_identity_sha256"] for row in (code, configs, tools)}})
environment_lock = discover_unique_file(configs["materialized_root"], "kaggle_t4x2_environment.lock.json")
requirements_root = discover_unique_file(configs["materialized_root"], "kaggle_base.lock").parent
wheel_root = pathlib.Path("/kaggle/working/certvic_cp312_wheels")
output = pathlib.Path("/kaggle/working/certvic_offline_wheelhouse_cp312.zip")
result = deterministic_provision(wheel_root=wheel_root, output=output, requirements_root=requirements_root, profile_id="kaggle_cp312_2026_07", environment_lock=environment_lock)
print(json.dumps(result, indent=2, sort_keys=True))


In [ ]:
from certvic.cvpr.environment_lock import prepare_offline_environment, select_locked_runtime
from certvic.cvpr.kaggle_bundle import verify_bundle
verification = verify_bundle(output)
if not verification["passed"] or result.get("deterministic_rebuild", {}).get("byte_identical") is not True:
    raise RuntimeError("CP312 wheelhouse deterministic validation failed")
validation_root = pathlib.Path("/kaggle/working/certvic_cp312_offline_validation")
if validation_root.exists(): shutil.rmtree(validation_root)
with zipfile.ZipFile(output) as archive: archive.extractall(validation_root)
selected_profile = select_locked_runtime(environment_lock)
offline_validation = prepare_offline_environment(environment_lock, wheelhouse=validation_root / "wheels", wheelhouse_manifest=validation_root / "wheelhouse_manifest.json", allow_preinstalled=False, require_exact=True, require_cuda=False, selected_profile=selected_profile, venv_root=pathlib.Path("/kaggle/working/certvic_runtime/kaggle_cp312_builder_validation"))
print({"resolver_result": result.get("resolver_result"), "supported_tags": selected_profile["observed_runtime"]["supported_tags"], "wheel_hashes": {name: row["sha256"] for name, row in offline_validation["wheelhouse_validation"]["files"].items()}, "offline_install_import_validation": offline_validation})
print({"status": "CP312_WHEELHOUSE_BUILDER_READY", "runtime_profile": "kaggle_cp312_2026_07", "bundle_sha256": verification["sha256"], "size": output.stat().st_size, "wheel_count": result.get("wheel_count"), "deterministic_rebuild": result["deterministic_rebuild"], "offline_validation_status": offline_validation["status"], "network_used_for_provisioning": True, "paper_evidence": False})
print("NEXT: download certvic_offline_wheelhouse_cp312.zip, upload it as a private Kaggle dataset, then start a fresh 00A session with Accelerator OFF and Internet OFF.")
